# 02aii · Cell type annotation (**Cross-organ ILCs** analytical manifold, acrosslifespan/no nEMC)

**Sections:**
1. Load integrated .h5ad object from `02ai_scvi_integration_crossorgan_ILC_acrosslifespan_noEMC.ipynb`
2. View existing metadata via UMAP
3. Plot previous cell type annotations
4. Optional: re-cluster specific Leiden clusters
5. TF-IDF: identify quick markers across Leiden clusters
6. Visualise known markers
7. Manually map cell types to Leiden clusters along with accompanying cell ontology
8. Save .h5ad
9. Contextualise cell type annotations within metadata with bar plots

*Note:* Several rounds of iteration within Sections 4-6 was performed prior to finalising cell type annotations.

## Configuration

In [ ]:
import os, sys

# ── Paths ─────────────────────────────────────────────────────────────────────
OBJECT            = "crossdisease_ILC"
MAIN_DIR          = "/nfs/team292/projects/PanTissue/"
INPUT_H5AD        = os.path.join(MAIN_DIR, f"results/temp/01_integration/integrated_scvi_{OBJECT}.h5ad")
OUTPUT_DIR        = os.path.join(MAIN_DIR, "results/temp/02_annotation/immune/acrosslifespan/noHormones/04_crossdisease_ILC/")
FORANNOTION_H5AD  = os.path.join(OUTPUT_DIR, f"forannotation_{OBJECT}.h5ad")
ANNOTATED_H5AD    = os.path.join(OUTPUT_DIR, f"annotated_{OBJECT}.h5ad")
ONTOLOGY_CSV      = os.path.join(MAIN_DIR, "data/freeze/ontology/HCA_annotation_ontology_L1_L4_immune.csv")

MARKERS_PY        = os.path.join(MAIN_DIR, "code/working/ck18/utils/markers.py")
MARKERS_IMMUNE_PY = os.path.join(MAIN_DIR, "code/working/ck18/utils/00_KeyMarkers_acrosslifespan_noHormones_immune.py")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Column names in obs ───────────────────────────────────────────────────────
DATASET_COL  = "Dataset"
DONOR_COL    = "Donor_id"
SAMPLE_COL   = "sample_id"
TISSUE_COL   = "Tissue"
LEIDEN_COL   = "leiden"

# ── Misc ──────────────────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── TF-IDF parameters ─────────────────────────────────────────────────────────
TFIDF_DOWNSAMPLE_N = 1000
TFIDF_N_MARKERS    = 10
TFIDF_FDR          = 0.01
TFIDF_EXPRESS_CUT  = 0.9

## Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import importlib.util
import numpy as np
import math
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.ticker as mticker
import seaborn as sns
import anndata as ad
import scanpy as sc

sys.path.append('/nfs/team292/projects/PanTissue/code/working/lg18/utils/')
from panatlas_utils import quick_markers, Barplot, filterTFIDF, plotUMAP_per_value, get_available_markers, plot_obs_facets
# sys.path.append('/nfs/team292/ck18/pipelines/')
# # from sc_analysis_ck18 import *
# from sc_plotting_ck18 import *

sc.settings.figdir = os.path.join(OUTPUT_DIR, "figures/")
sc.settings.verbosity = 3
sc.set_figure_params(dpi=100, frameon=False, figsize=(6, 5))

print(f"anndata : {ad.__version__}")
print(f"scanpy  : {sc.__version__}")

# 1. Load integrated object

In [ ]:
adata = ad.read_h5ad(INPUT_H5AD)
# adata = ad.read_h5ad(ANNOTATED_H5AD)
print(adata)
print("\nobs columns:", adata.obs.columns.tolist())

In [ ]:
adata.obs.head(1)

# 2. Overview UMAPs

In [ ]:
sc.pl.umap(
    adata,
    color           = LEIDEN_COL,
    legend_loc      = "on data",
    legend_fontsize = 10,
    save            = f"_02_leiden_{OBJECT}.png"
)

In [ ]:
sc.pl.umap(
    adata,
    color           = "phase",
    # legend_loc      = "on data",
    legend_fontsize = 10
)

### Annotations

In [ ]:
# Columns to merge
COL_1     = "immune_annotations"
COL_2     = "celltype_HCA"
MERGE_COL = "ANNOTATIONS_MERGED"

both_filled = (
    adata.obs[COL_1].notna() &
    adata.obs[COL_2].notna()
)

if int(both_filled.sum()) != 0:
    print(f"WARNING: Values in '{COL_1}' and '{COL_2}' are not mutually exclusive. Do NOT proceed with merging.")
else:
    adata.obs[MERGE_COL] = (
    adata.obs[COL_1]
    .combine_first(adata.obs[COL_2])
)
    print(adata.obs[MERGE_COL].value_counts())

In [ ]:
# Plot merged annotations
sc.pl.umap(
    adata,
    color           = MERGE_COL,
    # legend_loc      = "on data",
    legend_fontsize = 11,
    save            = f"_01_ANNOTATIONS_MERGED_{OBJECT}.png",
)

In [ ]:
plot_obs_facets(adata, MERGE_COL, n_cols=5)

### Tissue

In [ ]:
adata.obs.rename(columns={"Tissue": "Tissue_vl6"}, inplace=True)

In [ ]:
# —— Create new Tissue column (HCA) ———————————————————————————————————————————————————
adata.obs["Tissue"] = np.nan

organ_mask = adata.obs["Organ"].isin([
    "Ovary",
    "Fallopian Tube",
    "Cervix",
    "Vagina",
    "Mesonephros",
    "Reproductive tract"
])

adata.obs.loc[organ_mask, "Tissue"] = adata.obs.loc[organ_mask, "Organ"]

# 1. Endometrium
endo_mask = adata.obs["Tissue_ROI"].isin([
    "Endometrium",
    "Superficial_Endometrium"
])

adata.obs.loc[endo_mask, "Tissue"] = "Endometrium"

# 2. Myometrium
myo_mask = adata.obs["Tissue_ROI"].eq("Myometrium")
adata.obs.loc[myo_mask, "Tissue"] = "Myometrium"

# 3. Menstrual fluid
mf_mask = adata.obs["Tissue_ROI"].isin([
    "Menstrual fluid",
    "Mentrual fluid"   # catches typo variant present in your data
])

adata.obs.loc[mf_mask, "Tissue"] = "Menstrual fluid"

# 4. Whole Uterus
wu_mask = adata.obs["Tissue_ROI"].eq("Whole_Uterus")
adata.obs.loc[wu_mask, "Tissue"] = "Whole Uterus"

adata.obs["Tissue"].value_counts()

In [ ]:
# Columns to merge
COL_1     = "Tissue_vl6"
COL_2     = "Tissue"
MERGE_COL = "TISSUE_MERGED"

both_filled = (
    adata.obs[COL_1].notna() &
    adata.obs[COL_2].notna()
)

if int(both_filled.sum()) != 0:
    print(f"WARNING: Values in '{COL_1}' and '{COL_2}' are not mutually exclusive. Do NOT proceed with merging.")
else:
    adata.obs[MERGE_COL] = (
    adata.obs[COL_1]
    .combine_first(adata.obs[COL_2])
)
    print(adata.obs[MERGE_COL].value_counts())

In [ ]:
# Plot merged annotations
sc.pl.umap(
    adata,
    color           = MERGE_COL,
    # legend_loc      = "on data",
    legend_fontsize = 11,
    save            = f"_01_TISSUE_MERGED_{OBJECT}.png",
)

In [ ]:
plot_obs_facets(adata, MERGE_COL, n_cols=5)

### Disease

In [ ]:
# —— Rename values in existing disease columns ———————————————————————————————————————————————————
adata.obs['Disease'] = adata.obs['Disease'].replace(
    "control",
    "Healthy"
)

In [ ]:
# Columns to merge
COL_1     = "Patient_status"
COL_2     = "Disease"
MERGE_COL = "DISEASE_MERGED"

both_filled = (
    adata.obs[COL_1].notna() &
    adata.obs[COL_2].notna()
)

if int(both_filled.sum()) != 0:
    print(f"WARNING: Values in '{COL_1}' and '{COL_2}' are not mutually exclusive. Do NOT proceed with merging.")
else:
    adata.obs[MERGE_COL] = (
    adata.obs[COL_1]
    .combine_first(adata.obs[COL_2])
)
    print(adata.obs[MERGE_COL].value_counts())

In [ ]:
# Plot merged annotations
sc.pl.umap(
    adata,
    color           = MERGE_COL,
    # legend_loc      = "on data",
    legend_fontsize = 11,
    save            = f"_01_{MERGE_COL}_{OBJECT}.png",
)

### Developmental_stage

In [ ]:
# Fix typos
adata.obs["Menstrual_stage"] = (
    adata.obs["Menstrual_stage"]
    .replace("Unkown", "Unknown")
    .astype("category") # Ensure it stays/becomes a category for Scanpy
)
# Fix typos
adata.obs["Menstrual_stage"] = (
    adata.obs["Menstrual_stage"]
    .replace("Postmenopausal", "Postmenopause")
    .astype("category") # Ensure it stays/becomes a category for Scanpy
)
# Fix typos
adata.obs["Menstrual_stage"] = (
    adata.obs["Menstrual_stage"]
    .replace("Secreotry", "Secretory")
    .astype("category") # Ensure it stays/becomes a category for Scanpy
)
# Fix typos
adata.obs["Organ_part"] = (
    adata.obs["Organ_part"]
    .replace("Mentrual fluid", "Menstrual fluid")
    .astype("category") # Ensure it stays/becomes a category for Scanpy
)
# Fix typos
adata.obs["Menstrual_stage_short"] = (
    adata.obs['Menstrual_stage'].astype(str).str.split(" ").str[0]
)
adata.obs["Menstrual_stage_short"] = (
    adata.obs["Menstrual_stage_short"]
    .replace("Not", "Not applicable")
)

print(adata.obs["Menstrual_stage_short"].value_counts())

# Rename menstrual fluid
mask = (adata.obs['dataset'] == "uterus_adult_menstrualfluid_sanger-denoised") & (adata.obs['Menstrual_stage_short'] == "Menstrual")
adata.obs.loc[mask, 'Menstrual_stage_short'] = "Menstrual_fluid"

In [ ]:
# Create Developmental_stage_v2 column
adata.obs["Developmental_stage_v2"] = np.nan

mask_postmeno = adata.obs["Menstrual_stage_short"].eq("Postmenopause")
adata.obs.loc[mask_postmeno, "Developmental_stage_v2"] = "Postmenopausal"

mask_fetal = adata.obs["Developmental_stage"].eq("Fetal")
adata.obs.loc[mask_fetal & adata.obs["Developmental_stage_v2"].isna(),
              "Developmental_stage_v2"] = "Fetal"

mask_ped = adata.obs["Developmental_stage"].isin(["paediatric", "pubertal"])
adata.obs.loc[mask_ped & adata.obs["Developmental_stage_v2"].isna(),
              "Developmental_stage_v2"] = "Pediatric"

mask_rep = (
    adata.obs["Developmental_stage"].isin(["Adult", "postpubertal"]) &
    ~adata.obs["Menstrual_stage_short"].eq("Postmenopause")
)

adata.obs.loc[mask_rep & adata.obs["Developmental_stage_v2"].isna(),
              "Developmental_stage_v2"] = "Rep_age"

adata.obs["Developmental_stage_v2"].value_counts()

In [ ]:
# Developmental stage colors
COL = "Developmental_stage_v2"

colors = {
    "Fetal": "#4db6ac",
    "Pediatric": "#d4a373",
    "Rep_age": "#9575cd",
    "Postmenopausal": "#90a4ae"
}

adata.obs[COL] = adata.obs[COL].astype("category")

adata.obs[COL] = adata.obs[COL].cat.reorder_categories(
    colors.keys(),
    ordered=True
)

adata.uns[f"{COL}_colors"] = [colors[cat] for cat in adata.obs[COL].cat.categories]

In [ ]:
sc.pl.umap(
    adata,
    color           = "Developmental_stage_v2",
    ncols           = 1,
    legend_loc      = "right margin",
    legend_fontsize = 11,
    title           = "Developmental stage",
    # save            = f"_03_Dev_stage_acrosslifespan_noHormones_immune.pdf",
)

# 3. Plot previous annotations

In [ ]:
# ── Prior annotation (optional) ───────────────────────────────────────────────
# Maps a column from a prior annotation CSV into adata.obs (no subsetting).
# The CSV must have a 'barcode' index column. Set to None to skip.
PRIOR_ANN_CSV = os.path.join(f"/nfs/team292/projects/PanTissue/results/temp/02_annotation/immune/acrosslifespan/noHormones/20260514_acrosslifespan_noHormones_immune_annotations.csv")
PRIOR_ANN_COL = "celltype_HCA"
PRIOR_BAR_COL = "barcode"

# ── Map prior annotation to obs ───────────────────────────────────────────────
# Reads PRIOR_ANN_CSV and maps PRIOR_ANN_COL into adata.obs by barcode.
# Cells not present in the CSV receive NaN. No subsetting is performed.

if PRIOR_ANN_CSV and os.path.exists(PRIOR_ANN_CSV):
    prior = pd.read_csv(PRIOR_ANN_CSV, index_col=PRIOR_BAR_COL)
    # prior.index = prior.index.astype(str) + "-1"
    if PRIOR_ANN_COL in prior.columns:
        adata.obs[PRIOR_ANN_COL] = adata.obs_names.map(prior[PRIOR_ANN_COL])
        n_mapped = adata.obs[PRIOR_ANN_COL].notna().sum()
        print(f"Mapped '{PRIOR_ANN_COL}' from {PRIOR_ANN_CSV}")
        print(f"  {n_mapped:,} / {adata.n_obs:,} cells annotated")
        print(adata.obs[PRIOR_ANN_COL].value_counts(dropna=False))
    else:
        print(f"WARNING: column '{PRIOR_ANN_COL}' not found in {PRIOR_ANN_CSV}")
        print(f"  Available columns: {prior.columns.tolist()}")
elif PRIOR_ANN_CSV:
    print(f"WARNING: prior annotation CSV not found:\n  {PRIOR_ANN_CSV}")
else:
    print("No prior annotation CSV specified (PRIOR_ANN_CSV = None) — skipping.")

# Rename
adata.obs = adata.obs.rename(columns={PRIOR_ANN_COL: 'celltype_HCA'})

In [ ]:
# 3. Cast to category for plotting
sc.pl.umap(
    adata,
    color           = "celltype_HCA",
    # legend_loc      = "on data",
    legend_fontsize = 7,
    title           = "Cell type",
    # save            = "_04_HCA_celltype_label_acrosslifespan_noHormones_immune_legend.pdf",
)

In [ ]:
# Existing annotations
sc.pl.umap(
    adata,
    color           = "immune_annotations",
    # legend_loc      = "on data",
    legend_fontsize = 7,
)

# 4. Optional: customise Leiden clusters

### Re-cluster/merge clusters

In [ ]:
adata.obs["leiden_orig"] = adata.obs[LEIDEN_COL]

In [ ]:
# ── TEMPLATE: split leiden clusters ───────────────—————————————————————————
sc.tl.leiden(adata, resolution=0.3, restrict_to=(LEIDEN_COL, ['11']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[f'{LEIDEN_COL}_R'] = adata.obs[f'{LEIDEN_COL}_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs[f'{LEIDEN_COL}_R'].cat.categories))]
)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f'{LEIDEN_COL}_R']

In [ ]:
# ── TEMPLATE: split leiden clusters ───────────────—————————————————————————
sc.tl.leiden(adata, resolution=0.3, restrict_to=(LEIDEN_COL, ['15']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[f'{LEIDEN_COL}_R'] = adata.obs[f'{LEIDEN_COL}_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs[f'{LEIDEN_COL}_R'].cat.categories))]
)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f'{LEIDEN_COL}_R']

In [ ]:
# ── TEMPLATE: split leiden clusters ───────────────—————————————————————————
sc.tl.leiden(adata, resolution=0.3, restrict_to=(LEIDEN_COL, ['6']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[f'{LEIDEN_COL}_R'] = adata.obs[f'{LEIDEN_COL}_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs[f'{LEIDEN_COL}_R'].cat.categories))]
)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f'{LEIDEN_COL}_R']

In [ ]:
# ── TEMPLATE: split leiden clusters ───────────────—————————————————————————
sc.tl.leiden(adata, resolution=0.3, restrict_to=(LEIDEN_COL, ['13']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[f'{LEIDEN_COL}_R'] = adata.obs[f'{LEIDEN_COL}_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs[f'{LEIDEN_COL}_R'].cat.categories))]
)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f'{LEIDEN_COL}_R']

In [ ]:
# ── TEMPLATE: split leiden clusters ───────────────—————————————————————————
sc.tl.leiden(adata, resolution=0.3, restrict_to=(LEIDEN_COL, ['2']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[f'{LEIDEN_COL}_R'] = adata.obs[f'{LEIDEN_COL}_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs[f'{LEIDEN_COL}_R'].cat.categories))]
)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f'{LEIDEN_COL}_R']

In [ ]:
# ── TEMPLATE: merge leiden clusters ───────────────—————————————————————————
CLUSTERS_TO_MERGE     = [8,9,10]

CLUSTERS_TO_MERGE_STR = [str(c) for c in CLUSTERS_TO_MERGE]

# Replace every cluster in the merge list with the first value in the list
merge_target = CLUSTERS_TO_MERGE_STR[0]
adata.obs[f"{LEIDEN_COL}_R"] = adata.obs[f"{LEIDEN_COL}_R"].replace(
    {cluster: merge_target for cluster in CLUSTERS_TO_MERGE_STR}
)

# ── Step 2: Recode all clusters as consecutive integers (0, 1, 2, …) ─────────

current_categories = sorted(
    adata.obs[f"{LEIDEN_COL}_R"].unique(),
    key=lambda x: int(x)
)

recode_map = {old: str(new_idx) for new_idx, old in enumerate(current_categories)}

adata.obs[f"{LEIDEN_COL}_R"] = (
    adata.obs[f"{LEIDEN_COL}_R"]
    .map(recode_map)
    .astype("category")
)

sc.pl.umap(adata, color=f"{LEIDEN_COL}_R", legend_loc="on data", legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f"{LEIDEN_COL}_R"]

In [ ]:
# ── TEMPLATE: merge leiden clusters ───────────────—————————————————————————
CLUSTERS_TO_MERGE     = [14,15]

CLUSTERS_TO_MERGE_STR = [str(c) for c in CLUSTERS_TO_MERGE]

# Replace every cluster in the merge list with the first value in the list
merge_target = CLUSTERS_TO_MERGE_STR[0]
adata.obs[f"{LEIDEN_COL}_R"] = adata.obs[f"{LEIDEN_COL}_R"].replace(
    {cluster: merge_target for cluster in CLUSTERS_TO_MERGE_STR}
)

# ── Step 2: Recode all clusters as consecutive integers (0, 1, 2, …) ─────────

current_categories = sorted(
    adata.obs[f"{LEIDEN_COL}_R"].unique(),
    key=lambda x: int(x)
)

recode_map = {old: str(new_idx) for new_idx, old in enumerate(current_categories)}

adata.obs[f"{LEIDEN_COL}_R"] = (
    adata.obs[f"{LEIDEN_COL}_R"]
    .map(recode_map)
    .astype("category")
)

sc.pl.umap(adata, color=f"{LEIDEN_COL}_R", legend_loc="on data", legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f"{LEIDEN_COL}_R"]

In [ ]:
# ── TEMPLATE: merge leiden clusters ───────────────—————————————————————————
CLUSTERS_TO_MERGE     = [13,15]

CLUSTERS_TO_MERGE_STR = [str(c) for c in CLUSTERS_TO_MERGE]

# Replace every cluster in the merge list with the first value in the list
merge_target = CLUSTERS_TO_MERGE_STR[0]
adata.obs[f"{LEIDEN_COL}_R"] = adata.obs[f"{LEIDEN_COL}_R"].replace(
    {cluster: merge_target for cluster in CLUSTERS_TO_MERGE_STR}
)

# ── Step 2: Recode all clusters as consecutive integers (0, 1, 2, …) ─────────

current_categories = sorted(
    adata.obs[f"{LEIDEN_COL}_R"].unique(),
    key=lambda x: int(x)
)

recode_map = {old: str(new_idx) for new_idx, old in enumerate(current_categories)}

adata.obs[f"{LEIDEN_COL}_R"] = (
    adata.obs[f"{LEIDEN_COL}_R"]
    .map(recode_map)
    .astype("category")
)

sc.pl.umap(adata, color=f"{LEIDEN_COL}_R", legend_loc="on data", legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f"{LEIDEN_COL}_R"]

In [ ]:
# ── TEMPLATE: merge leiden clusters ───────────────—————————————————————————
CLUSTERS_TO_MERGE     = [13,14,15]

CLUSTERS_TO_MERGE_STR = [str(c) for c in CLUSTERS_TO_MERGE]

# Replace every cluster in the merge list with the first value in the list
merge_target = CLUSTERS_TO_MERGE_STR[0]
adata.obs[f"{LEIDEN_COL}_R"] = adata.obs[f"{LEIDEN_COL}_R"].replace(
    {cluster: merge_target for cluster in CLUSTERS_TO_MERGE_STR}
)

# ── Step 2: Recode all clusters as consecutive integers (0, 1, 2, …) ─────────

current_categories = sorted(
    adata.obs[f"{LEIDEN_COL}_R"].unique(),
    key=lambda x: int(x)
)

recode_map = {old: str(new_idx) for new_idx, old in enumerate(current_categories)}

adata.obs[f"{LEIDEN_COL}_R"] = (
    adata.obs[f"{LEIDEN_COL}_R"]
    .map(recode_map)
    .astype("category")
)

sc.pl.umap(adata, color=f"{LEIDEN_COL}_R", legend_loc="on data", legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f"{LEIDEN_COL}_R"]

In [ ]:
# ── TEMPLATE: merge leiden clusters ───────────────—————————————————————————
CLUSTERS_TO_MERGE     = [2,4,5]

CLUSTERS_TO_MERGE_STR = [str(c) for c in CLUSTERS_TO_MERGE]

# Replace every cluster in the merge list with the first value in the list
merge_target = CLUSTERS_TO_MERGE_STR[0]
adata.obs[f"{LEIDEN_COL}_R"] = adata.obs[f"{LEIDEN_COL}_R"].replace(
    {cluster: merge_target for cluster in CLUSTERS_TO_MERGE_STR}
)

# ── Step 2: Recode all clusters as consecutive integers (0, 1, 2, …) ─────────

current_categories = sorted(
    adata.obs[f"{LEIDEN_COL}_R"].unique(),
    key=lambda x: int(x)
)

recode_map = {old: str(new_idx) for new_idx, old in enumerate(current_categories)}

adata.obs[f"{LEIDEN_COL}_R"] = (
    adata.obs[f"{LEIDEN_COL}_R"]
    .map(recode_map)
    .astype("category")
)

sc.pl.umap(adata, color=f"{LEIDEN_COL}_R", legend_loc="on data", legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f"{LEIDEN_COL}_R"]

In [ ]:
# ── TEMPLATE: split leiden clusters ───────────────—————————————————————————
sc.tl.leiden(adata, resolution=0.3, restrict_to=(LEIDEN_COL, ['9']),
             flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[f'{LEIDEN_COL}_R'] = adata.obs[f'{LEIDEN_COL}_R'].cat.rename_categories(
    [str(i) for i in range(len(adata.obs[f'{LEIDEN_COL}_R'].cat.categories))]
)
sc.pl.umap(adata, color=f'{LEIDEN_COL}_R', legend_loc='on data', legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f'{LEIDEN_COL}_R']

In [ ]:
# ── TEMPLATE: merge leiden clusters ───────────────—————————————————————————
CLUSTERS_TO_MERGE     = [8,9,10,11,16]

CLUSTERS_TO_MERGE_STR = [str(c) for c in CLUSTERS_TO_MERGE]

# Replace every cluster in the merge list with the first value in the list
merge_target = CLUSTERS_TO_MERGE_STR[0]
adata.obs[f"{LEIDEN_COL}_R"] = adata.obs[f"{LEIDEN_COL}_R"].replace(
    {cluster: merge_target for cluster in CLUSTERS_TO_MERGE_STR}
)

# ── Step 2: Recode all clusters as consecutive integers (0, 1, 2, …) ─────────

current_categories = sorted(
    adata.obs[f"{LEIDEN_COL}_R"].unique(),
    key=lambda x: int(x)
)

recode_map = {old: str(new_idx) for new_idx, old in enumerate(current_categories)}

adata.obs[f"{LEIDEN_COL}_R"] = (
    adata.obs[f"{LEIDEN_COL}_R"]
    .map(recode_map)
    .astype("category")
)

sc.pl.umap(adata, color=f"{LEIDEN_COL}_R", legend_loc="on data", legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f"{LEIDEN_COL}_R"]

In [ ]:
# ── TEMPLATE: merge leiden clusters ───────────────—————————————————————————
CLUSTERS_TO_MERGE     = [4,5]

CLUSTERS_TO_MERGE_STR = [str(c) for c in CLUSTERS_TO_MERGE]

# Replace every cluster in the merge list with the first value in the list
merge_target = CLUSTERS_TO_MERGE_STR[0]
adata.obs[f"{LEIDEN_COL}_R"] = adata.obs[f"{LEIDEN_COL}_R"].replace(
    {cluster: merge_target for cluster in CLUSTERS_TO_MERGE_STR}
)

# ── Step 2: Recode all clusters as consecutive integers (0, 1, 2, …) ─────────

current_categories = sorted(
    adata.obs[f"{LEIDEN_COL}_R"].unique(),
    key=lambda x: int(x)
)

recode_map = {old: str(new_idx) for new_idx, old in enumerate(current_categories)}

adata.obs[f"{LEIDEN_COL}_R"] = (
    adata.obs[f"{LEIDEN_COL}_R"]
    .map(recode_map)
    .astype("category")
)

sc.pl.umap(adata, color=f"{LEIDEN_COL}_R", legend_loc="on data", legend_fontsize=9)

adata.obs[LEIDEN_COL] = adata.obs[f"{LEIDEN_COL}_R"]

### Final leiden clusters

In [ ]:
adata.obs[LEIDEN_COL].unique()

In [ ]:
plot_obs_facets(adata, LEIDEN_COL, n_cols=6)

In [ ]:
sc.pl.umap(
    adata,
    color           = LEIDEN_COL,
    legend_loc      = "on data",
    legend_fontsize = 10,
    legend_fontoutline = 0.75,
    title           = "Leiden_RECLUSTERED",
    palette         = palette,
    save            = f"_03_leidenR_{OBJECT}.png",
)

# 5. TF-IDF: quick marker identification

### All cells TF-IDF

In [ ]:
ANNOT_COL = "leiden"

In [ ]:
cells_down = (
    adata.obs
    .groupby(ANNOT_COL, observed=True)
    .apply(lambda g: g.sample(min(len(g), TFIDF_DOWNSAMPLE_N), random_state=RANDOM_SEED))
    .index.get_level_values(-1)
)
adata_down = adata[cells_down].copy()
print(f"Downsampled adata: {adata_down.n_obs} cells across {adata_down.obs[ANNOT_COL].nunique()} groups")

In [ ]:
tfidf_markers = quick_markers(
    adata_down,
    cluster_key  = ANNOT_COL,
    n_markers    = 200,
    fdr          = TFIDF_FDR,
    express_cut  = TFIDF_EXPRESS_CUT,
)

TFIDF_CSV = os.path.join(OUTPUT_DIR, f"tfidf_markers_{ANNOT_COL}_{OBJECT}.csv")
tfidf_markers.to_csv(TFIDF_CSV, index=False)
print(f"TF-IDF markers saved → {TFIDF_CSV}")
tfidf_markers.head(5)

In [ ]:
lst_tfidf_markers = filterTFIDF(tfidf_markers, N=20)
sc.pl.dotplot(
    adata_down,
    lst_tfidf_markers,
    groupby        = ANNOT_COL,
    standard_scale = 'var',
    save           = f'tfidf_markers_{ANNOT_COL}_{OBJECT}.pdf',
)


### Subset TF-IDF

In [ ]:
ANNOT_COL = "celltype"
GROUP_COL = "TISSUE_BROAD"

In [ ]:
to_keep = [
    "ILC3_NCRhi"
]

bdata = adata[adata.obs[ANNOT_COL].isin(to_keep)].copy()
print(f"Num cells removed: {(adata.n_obs)-(bdata.n_obs)}")
print(f"Num cells remaining: {bdata.n_obs}")
bdata.obs[ANNOT_COL].value_counts()

In [ ]:
cells_down = (
    bdata.obs
    .groupby(GROUP_COL, observed=True)
    .apply(lambda g: g.sample(min(len(g), TFIDF_DOWNSAMPLE_N), random_state=RANDOM_SEED))
    .index.get_level_values(-1)
)
bdata_down = bdata[cells_down].copy()
print(f"Downsampled adata: {bdata_down.n_obs} cells across {bdata_down.obs[ANNOT_COL].nunique()} groups")

In [ ]:
tfidf_markers = quick_markers(
    bdata_down,
    cluster_key  = GROUP_COL,
    n_markers    = 50,
    fdr          = TFIDF_FDR,
    express_cut  = TFIDF_EXPRESS_CUT,
)

TFIDF_CSV = os.path.join(OUTPUT_DIR, f"tfidf_markers_ILC3_NCRhi_{GROUP_COL}_{OBJECT}.csv")
tfidf_markers.to_csv(TFIDF_CSV, index=False)
print(f"TF-IDF markers saved → {TFIDF_CSV}")
tfidf_markers.head(5)

In [ ]:
lst_tfidf_markers = filterTFIDF(tfidf_markers, N=20)
sc.pl.dotplot(
    bdata,
    lst_tfidf_markers,
    groupby        = GROUP_COL,
    standard_scale = 'var',
    save           = f'tfidf_markers_ILC3_NCRhi_{GROUP_COL}_{OBJECT}.pdf',
)


In [ ]:
ANNOT_COL = "celltype"
GROUP_COL = "TISSUE_MERGED"

In [ ]:
to_keep = [
    "ILC3_NCRhi"
]

bdata = adata[adata.obs[ANNOT_COL].isin(to_keep)].copy()
print(f"Num cells removed: {(adata.n_obs)-(bdata.n_obs)}")
print(f"Num cells remaining: {bdata.n_obs}")
bdata.obs[ANNOT_COL].value_counts()

In [ ]:
cells_down = (
    bdata.obs
    .groupby(GROUP_COL, observed=True)
    .apply(lambda g: g.sample(min(len(g), TFIDF_DOWNSAMPLE_N), random_state=RANDOM_SEED))
    .index.get_level_values(-1)
)
bdata_down = bdata[cells_down].copy()
print(f"Downsampled adata: {bdata_down.n_obs} cells across {bdata_down.obs[ANNOT_COL].nunique()} groups")

In [ ]:
tfidf_markers = quick_markers(
    bdata_down,
    cluster_key  = GROUP_COL,
    n_markers    = 50,
    fdr          = TFIDF_FDR,
    express_cut  = TFIDF_EXPRESS_CUT,
)

TFIDF_CSV = os.path.join(OUTPUT_DIR, f"tfidf_markers_ILC3_NCRhi_{GROUP_COL}_{OBJECT}.csv")
tfidf_markers.to_csv(TFIDF_CSV, index=False)
print(f"TF-IDF markers saved → {TFIDF_CSV}")
tfidf_markers.head(5)

In [ ]:
lst_tfidf_markers = filterTFIDF(tfidf_markers, N=20)
sc.pl.dotplot(
    bdata,
    lst_tfidf_markers,
    groupby        = GROUP_COL,
    standard_scale = 'var',
    save           = f'tfidf_markers_ILC3_NCRhi_{GROUP_COL}_{OBJECT}.pdf',
)


In [ ]:
to_keep = [
    "ILC3_NCRhi"
]

bdata = adata[adata.obs[ANNOT_COL].isin(to_keep)].copy()
print(f"Num cells removed: {(adata.n_obs)-(bdata.n_obs)}")
print(f"Num cells remaining: {bdata.n_obs}")
bdata.obs[ANNOT_COL].value_counts()

In [ ]:
# —— PLOT DEGs from ReproTract vs. SkinNM ———————————————————————————————————————————————————
DEG_genes = [
    "EPAS1",
    "CAMK4",
    "SLFN12L",
    "PLXDC2",
    "ST6GALNAC3",
    "SORCS1",
    "FNDC3B",
    "MAN2A1",
    "HIPK2",
    "TMEM156",
    "NEO1",
    "TENT5C",
    "SLC8A1",
    "TEX14",
    "AC012447.1",
    "GRAMD2B",
    "NTM",
    "LINC00910",
    "ALDOC",
    "GLB1",
    "ATF3",
    "CROCC",
    "CCL20",
]

markers = {
    "Repro_v_nonRepro upDEGs": DEG_genes,
    "Hypoxia": ['HIF1A']
}
sc.pl.dotplot(
    bdata,
    markers,
    groupby        = GROUP_COL,
    standard_scale = 'var'
)


# 6. Plot known markers

## Normalise for visualisation (raw counts frozen)

In [ ]:
adata.raw = adata   # freeze raw counts
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
sc.pp.log1p(adata)
print("Raw counts frozen in adata.raw. adata.X now holds log-normalised expression.")

## Check for doublets

In [ ]:
spec = importlib.util.spec_from_file_location("markers", MARKERS_PY)
markers_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(markers_mod)

In [ ]:
# ── 1. Broad lineage markers ──────────────────────────────────────────────────
markers_lineage = getattr(markers_mod, "markers_lineage", {})
markers_lineage_avail = get_available_markers(markers_lineage, "Lineage markers", adata)

In [ ]:
# ── Broad lineage markers (doublet / contamination check) ─────────────────────
sc.pl.dotplot(
    adata,
    var_names      = markers_lineage_avail,
    groupby        = LEIDEN_COL,
    use_raw        = True,
    standard_scale = "var"
)

In [ ]:
# —— Specific doublet markres ———————————————————————————————————————————————————
markers_doublets = {
    "T": ["CD3D"],
    "B": ["CD79A"],
    "Myeloid": ["SPI1", "CD68"]
}

In [ ]:
# ── 2. UMAP FEATURE PLOTS — confirm spatial distribution of key markers ────────
for k,v in markers_doublets.items():
    
    print(f"\n—— PLOTTING {k} markers {'—'*100}")
    
    genes_present = [g for g in v if g in adata.var_names]
    genes_missing = [g for g in v if g not in adata.var_names]
    
    if genes_missing:
        print(f"WARNING: the following marker genes were not found in adata.var_names:\n{genes_missing}")

    sc.pl.umap(
        adata,
        color   = genes_present,
        use_raw = False,
        ncols   = 4,
        frameon = False,
    )

## Donor-specific clusters

In [ ]:
sc.pl.umap(
    adata,
    color           = DONOR_COL,
    legend_loc      = "none",
    legend_fontsize = 4,
    palette         = palette
)

## Known markers

In [ ]:
# ── Curated ILC/NK marker gene panel ──────────────────────────────────────────—
# Organised by cell type, from broad lineage → fine subtypes
# Note: not all genes will be detected in every tissue or dataset

markers_ILCs = {

    # Broad lymphocyte / ILC lineage identity
    # ID2 and IL7R mark the common ILC progenitor lineage
    "General_ILC": [
        "ID2",      # Master ILC transcription factor; expressed across all ILCs
        "IL7R",     # CD127; marks ILCs broadly (low/absent on mature NK)
    ],

    # NK cells — cytotoxic, EOMES+, low/no IL7R
    "General_NK": [
        "EOMES",    # T-box TF; key discriminator — NK+ vs ILC1−
        "NCAM1",    # CD56; classical NK surface marker (also on some ILC1s)
        "NKG7",     # Granule protein; cytotoxicity marker
        "GNLY",     # Granulysin; cytotoxic effector
        "PRF1",     # Perforin; cytotoxic effector
        "KLRD1",    # CD94; NK receptor
        "KLRF1",    # NKp80; NK activation receptor
        "GZMB",     # Granzyme B; cytotoxicity (also in cytotoxic T cells)
        "GZMK",     # Granzyme K; marks less mature / CD56bright NK
    ],

    # Uterine/decidual NK (dNK) — distinct from peripheral NK
    # Key ref: Vento-Tormo et al. Nature 2018; Sharkey et al.
    "uNK": [
        "ITGA1",                         # CD49a; tissue-resident NK marker (also ILC1)
        "B4GALNT1", "KIR2DL1", "ENTPD1", # uNK1
        "CDHR1",                         # uNK2
        "CD160", "LDB2"                  # uNK3
    ],
    
    # Peripheral NK
    "pNK": [
        "FCGR3A",   # CD16; marks cytotoxic/peripheral NK (CD56dim subset)
        "FGFBP2",
        "SPON2",
        
        # CD56/16dim SPTSSBhi subset
        "SPTSSB",
        "TNFRSF11A",
        "TCF7",
        "SELL",
    ],

    # ILC1 — IFNg-producing, TBX21+, EOMES−
    # WARNING: ILC1 vs NK boundary is contested; use EOMES as key discriminator
    "ILC1": [
        "IL12RB2",  # IL-12 stimulates ILC1 to produce IFNG
        "TBX21",    # T-bet; shared with NK but required for ILC1
        "IFNG",     # IFN-gamma; key ILC1 effector cytokine (may be low at rest)
        "TNF",      # Effector cytokine (also expressed by NK)
        "CXCR3",    # Chemokine receptor; enriched on ILC1 vs ILC2/3
        "ZNF683",   # HOBIT; tissue-resident ILC1/NK marker
        # EOMES should be ABSENT or very low
    ],

    # ILC2 — type 2 immunity, GATA3-hi
    "ILC2": [
        "GATA3",    # Master ILC2 TF (also expressed at lower levels in ILC3)
        "PTGDR2",   # CRTH2; one of the most specific ILC2 surface markers
        "IL1RL1",   # ST2 (IL-33 receptor); highly specific for ILC2
        "HPGDS",    # Prostaglandin synthase; ILC2-enriched
        "RORA",     # Nuclear receptor; ILC2-associated TF
        
        "IL13",     # Type 2 cytokine (often not detected at rest in scRNA-seq)
        "IL5",      # Type 2 cytokine (often low at rest)
        "KLRB1",    # CD161; expressed on ILC2 and ILC3 (not specific alone)
    ],

    # ILC3 — IL-22/IL-17 producing, RORgt+, mucosal immunity
    "ILC3": [
        "RORC",     # RORgamma-t; master ILC3 TF
        "KIT",      # CD117 (c-Kit); marks ILC3 and ILC precursors
        "AHR",      # Aryl hydrocarbon receptor TF; ILC3-associated
        "IL1R1",

        "IL22",     # Key ILC3 effector cytokine (often low at rest)
                
        "NCR2",     # NKp44; marks NCR+ ILC3 subtype (common in mucosa)
        "NCR1",     # NKp46; expressed on NCR+ ILC3 and NK
        "PCDH9",
        
        "CA10",     # NCR- ILC3 in reproductive tract
    ],
    
    "LTi-like": [   # LTi in adult
        "IL17A",    # ILC3 cytokine (more LTi-like ILC3s)
        "CXCR5",    # Marks LTi-like / follicular ILC3 subset
        "TNFSF11",  # RANKL; LTi marker
        "TNFSF4",
        "TNFSF8"
    ],

    # ILC precursors — expect mainly in mesonephros / fetal samples
    "ILC precursor": [
        "ID2",      # Shared with mature ILCs, but high in progenitors
        "NFIL3",    # E4BP4; ILC fate specification TF
        "ZBTB16",   # ZBTB16; common ILC progenitor marker
        "TOX",      # Upstream of ILC fate; also in NK progenitors
        "IL7R",     # CD127; maintained in progenitors
        "KIT",      # CD117; progenitor / ILC3 marker
    ],
    
    "Cycling": ["MKI67", "PCLAF"]
}

In [ ]:
# ── 1. DOTPLOT ──────────────────────————————————————————————————————————————————
ANNOT_COL = "leiden"

sc.pl.dotplot(
    adata,
    var_names      = markers_ILCs,
    groupby        = ANNOT_COL,
    standard_scale = "var",
    use_raw        = True,
    swap_axes      = False,
    save           = f"01_markers_ILCs_leiden_{OBJECT}.png"
)

In [ ]:
# ── 2. UMAP FEATURE PLOTS — confirm spatial distribution of key markers ────────
for k,v in markers_ILCs.items():
    
    print(f"\n—— PLOTTING {k} markers {'—'*100}")
    
    genes_present = [g for g in v if g in adata.var_names]
    genes_missing = [g for g in v if g not in adata.var_names]
    
    if genes_missing:
        print(f"WARNING: the following marker genes were not found in adata.var_names:\n{genes_missing}")

    sc.pl.umap(
        adata,
        color   = genes_present,
        use_raw = False,
        ncols   = 4,
        frameon = False,
    )

In [ ]:
# Weird markers
adhoc = ['TNFSF4', 'TNFRSF4', 'KIT', 'RORC', 'NCR2']
sc.pl.umap(adata, color=adhoc, use_raw=False)
sc.pl.dotplot(adata, var_names=adhoc, groupby='leiden', use_raw=True, standard_scale="var", swap_axes=True)

In [ ]:
# Adhoc markers
adhoc = ['IL17RB', 'IL1R1', 'CTLA4', 'IL4I1', 'MAF', 'SLC4A10', 'CCR6', 'IL17A']
sc.pl.umap(adata, color=adhoc, use_raw=False)
sc.pl.dotplot(adata, var_names=adhoc, groupby='leiden', use_raw=True, standard_scale="var", swap_axes=True)

In [ ]:
# ILC1 receptor markers
adhoc = ['IL15RA', 'IL18R1', 'IL18RAP', 'IL12RB1', 'IL12RB2']
sc.pl.umap(adata, color=adhoc, use_raw=False)
sc.pl.dotplot(adata, var_names=adhoc, groupby='leiden', use_raw=True, standard_scale="var", swap_axes=True)

In [ ]:
# ILC2 receptor markers
adhoc = ['IL9R', 'IL2RA', 'IL17RB', 'IL1RL1', 'CRLF2', 'NMUR1', 'PTGDR2']
sc.pl.umap(adata, color=adhoc, use_raw=False)
sc.pl.dotplot(adata, var_names=adhoc, groupby='leiden', use_raw=True, standard_scale="var", swap_axes=True)

In [ ]:
# ILC3 receptor markers
adhoc = ['IL23R', 'IL1R1', 'IL1R2', 'IL1RAP']
sc.pl.umap(adata, color=adhoc, use_raw=False)
sc.pl.dotplot(adata, var_names=adhoc, groupby='leiden', use_raw=True, standard_scale="var", swap_axes=True)

### Subset by dataset

In [ ]:
COL = "Dataset"

to_keep = [
    "tonsil_ILC_eg22"
]

bdata = adata[adata.obs[COL].isin(to_keep)].copy()

print(f"Num cells removed: {(adata.n_obs)-(bdata.n_obs)}")
print(f"Num cells kept: {bdata.n_obs}")
bdata.obs[LEIDEN_COL].value_counts()

In [ ]:
# —— Manual Prog/Gran/Blood markers ———————————————————————————————————————————————————
sc.pl.dotplot(
    bdata,
    var_names      = markers_ILCs,
    groupby        = LEIDEN_COL,
    # use_raw        = True,
    standard_scale = "var",
    # save           = "01_markersL2_acrosslifespan_noHormones_immune.pdf",
)

# 7. Manual cell type annotation

Fill in Leiden cluster IDs (integers) for each postnatal immune cell type.  
Leave empty lists `[]` for cell types not present or not yet identified.

In [ ]:
# ── Fill in Leiden IDs as integers ────────────────────────────────────────────
# Example:  'Epi_FT_Cil': [3, 7]
celltype_to_leiden = {
    'ILC3_NCRhi': [8],
    'ILC3_NCRlo/LTi': [9],
    
    'uNK1': [1],
    'uNK2': [4],
    'uNK3': [2,3],
    'NK_CD56dim_CD16dim_SPTSSBhi': [6],
    'NK_CD16hi': [7],
    
    'NK_Cyc': [0,12],
    
        'doublet': [5,10,11],
        # 'soup': [],
        # 'lowQC': [],
        # 'donor_specific': [],
        # 'unknown': [],
}


# Convert integers to strings
celltype_to_leiden = {k: [str(v) for v in vals] for k, vals in celltype_to_leiden.items()}

# ── Validate Leiden IDs ───────────────────────────────────────────────────────
_all_assigned = [lid for lids in celltype_to_leiden.values() for lid in lids]
_all_leiden   = sorted(adata.obs[LEIDEN_COL].unique().tolist())
_dups         = [x for x in _all_assigned if _all_assigned.count(x) > 1]
_unassigned   = [x for x in _all_leiden if x not in _all_assigned]

if _dups:
    print(f"WARNING — clusters assigned to multiple cell types: {sorted(set(_dups))}")
if _unassigned:
    print(f"INFO    — clusters not yet assigned              : {_unassigned}")
if not _dups and not _unassigned:
    print("All clusters assigned, no duplicates.")

In [ ]:
# Invert: leiden_id → celltype
leiden_to_celltype = {
    lid: celltype
    for celltype, lids in celltype_to_leiden.items()
    for lid in lids
}

adata.obs["celltype"] = (
    adata.obs[LEIDEN_COL]
    .map(leiden_to_celltype)
    .astype("category")
)

print("Celltype distribution:")
print(adata.obs["celltype"].value_counts())

In [ ]:
# Order categories
celltype_order = list(celltype_to_leiden.keys())

adata.obs["celltype"] = adata.obs["celltype"].cat.reorder_categories(
    celltype_order,
    ordered=True
)

print("Celltype distribution:")
print(adata.obs["celltype"].value_counts())

In [ ]:
# celltype_leiden = "<celltype>_<leiden>"
adata.obs["celltype_leiden"] = (
    adata.obs["celltype"].astype(str)
    + "_"
    + adata.obs[LEIDEN_COL].astype(str)
).astype("category")

print("Celltype_leiden categories:", sorted(adata.obs["celltype_leiden"].cat.categories.tolist()))

In [ ]:
sc.pl.umap(
    adata,
    color              = "celltype",
    legend_loc         = "on data",
    legend_fontsize    = 6,
    legend_fontoutline = 1,
    save               = f"_04_celltype_{OBJECT}.png"
)

In [ ]:
sc.pl.umap(
    adata,
    color              = "celltype",
    legend_fontsize    = 11,
    save               = f"_04_celltype_{OBJECT}_legend.png"
)

In [ ]:
# —— Map celltype to L3 resolution —————————————————————————————————————————————
celltype_to_L3 = {
    'ILC3_NCRhi': 'ILC3',
    'ILC3_NCRlo/LTi': 'ILC3',
    
    'uNK1': 'uNK',
    'uNK2': 'uNK',
    'uNK3': 'uNK',
    'NK_CD56dim_CD16dim_SPTSSBhi': 'pNK',
    'NK_CD16hi': 'pNK',
    'NK_Cyc': 'NK_Cyc',
    
        'doublet': None,
}

adata.obs["L3"] = adata.obs["celltype"].map(celltype_to_L3)
adata.obs["L3"].value_counts()

In [ ]:
sc.pl.umap(
    adata,
    color              = "L3",
    legend_loc         = "on data",
    legend_fontsize    = 6,
    legend_fontoutline = 1,
    na_in_legend       = False,
    save               = f"_05_L3_{OBJECT}.png"
)

In [ ]:
sc.pl.umap(
    adata,
    color              = "L3",
    legend_fontsize    = 11,
    na_in_legend       = False,
    save               = f"_05_L3_{OBJECT}_legend.png"
)

# 8. Save

In [ ]:
adata.obs.head(1)

In [ ]:
adata.write_h5ad(FORANNOTION_H5AD)
print(f"For Annotation object saved → {FORANNOTION_H5AD}")

In [ ]:
adata

In [ ]:
adata.write_h5ad(ANNOTATED_H5AD)
print(f"Annotated object saved → {ANNOTATED_H5AD}")

## Save obs for annotations

In [ ]:
# ── Save annotations ────────────────────────────────—————————————
adata.obs.to_csv(
    f'{OUTPUT_DIR}20260526_{OBJECT}_annotations.csv',
    index=True,
    index_label='barcode'
)

# 9. Barplots

In [ ]:
def barplot(
    which_var,
    adata,
    var              = "lineage",
    # --- figure geometry ---
    height           = 3,
    width            = 4,
    bar_height       = 0.8,         # thickness of each bar (0–1)
    # --- axes formatting ---
    xlabel           = "Proportion (%)",
    xtick_interval   = 10,          # spacing between x-axis ticks
    fontsize         = 8,
    # --- legend ---
    legend_cols      = 1,
    legend_x         = 1.02,        # bbox_to_anchor x
    legend_y         = 1.0,         # bbox_to_anchor y
    legend_fontsize  = 7,
    show_legend      = True,
    # --- output ---
    save_pdf         = False,
    save_png         = False,
    output_dir       = os.getcwd(),
    filename_prefix  = "",
    dpi              = 300,
):
    """
    Horizontal stacked bar chart showing percentage composition of `which_var`
    broken down by `var` (e.g. cell type proportions per tissue).

    Parameters
    ----------
    which_var       : str   Column in adata.obs that defines the stacked segments
                            (e.g. "celltype_HCA").
    adata           : AnnData
    var             : str   Column in adata.obs that defines the bars / rows
                            (e.g. "Tissue").
    height          : float Figure height in inches.
    width           : float Figure width in inches.
    bar_height      : float Bar thickness as a fraction of row spacing (0–1).
    xlabel          : str   X-axis label.
    xtick_interval  : int   Gap between x-axis tick marks.
    fontsize        : int   Font size for axis tick labels.
    legend_cols     : int   Number of columns in the legend.
    legend_x/y      : float Legend anchor position (in axes-fraction units).
    legend_fontsize : int   Font size for legend text.
    show_legend     : bool  Toggle legend visibility.
    save_pdf        : bool  Save figure as PDF.
    save_png        : bool  Save figure as PNG.
    output_dir      : str   Directory for saved files.
    filename_prefix : str   Optional prefix for the output filename.
    dpi             : int   Resolution for saved files.

    Returns
    -------
    fig, ax : matplotlib Figure and Axes objects for further customisation.
    """

    # ── 1. BUILD PERCENTAGE TABLE ──────────────────────────────────────────────
    # Cross-tabulate: rows = var categories, columns = which_var categories
    # normalize='index' makes each row sum to 100%
    plotdata = pd.crosstab(
        adata.obs[var],
        adata.obs[which_var],
        normalize="index"
    ) * 100

    # ── 2. RESPECT CATEGORY ORDER ─────────────────────────────────────────────
    # If var is a pandas Categorical, use its defined order (reversed so the
    # first category appears at the TOP of the horizontal bar chart).
    # Bug fix: the original forgot to assign the result of reorder_categories().
    if hasattr(adata.obs[var], "cat"):
        desired_order = adata.obs[var].cat.categories.tolist()[::-1]
        # Only reindex with categories that actually appear in the crosstab
        desired_order = [c for c in desired_order if c in plotdata.index]
        plotdata = plotdata.reindex(desired_order)

    # ── 3. RESOLVE COLOUR ORDER ───────────────────────────────────────────────
    # Pull colours from adata.uns
    # Build a dict {category_name: hex} first, then reindex to plotdata columns
    # so the colour order always matches the stacked segments regardless of how
    # the crosstab sorted its columns.
    uns_key = f"{which_var}_colors"

    if uns_key in adata.uns:
        categories   = adata.obs[which_var].cat.categories.tolist()
        uns_colors   = adata.uns[uns_key]                          # list of hex strings
        color_lookup = dict(zip(categories, uns_colors))           # {cell_type: hex}
        color_list   = [color_lookup.get(c, "#AAAAAA") for c in plotdata.columns]
    else:
        # Fallback: warn the user rather than failing silently
        import warnings
        warnings.warn(
            f"adata.uns['{uns_key}'] not found — plotting with matplotlib defaults. "
            f"Run sc.pl.umap(adata, color='{which_var}') first to register colours, "
            f"or set them manually with adata.uns['{uns_key}'].",
            UserWarning
        )
        color_list = None

    # ── 4. PLOT ───────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(width, height))

    plotdata.plot.barh(
        stacked    = True,
        ax         = ax,          # plot onto our axes (avoids figure duplication)
        edgecolor  = "none",
        color      = color_list,
        width      = bar_height,  # pandas barh uses 'width' for bar thickness
        legend     = False,       # we handle the legend ourselves below
        grid       = False
    )

    # ── 5. AXES FORMATTING ────────────────────────────────────────────────────
    ax.set_xticks(np.arange(0, 101, xtick_interval))
    ax.set_xlim(0, 100)
    ax.set_xlabel(xlabel, fontsize=fontsize)
    ax.set_ylabel(var, fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)

    # Remove top and right spines for a cleaner publication look
    # ax.spines["top"].set_visible(False)
    # ax.spines["right"].set_visible(False)

    # ── 6. LEGEND ─────────────────────────────────────────────────────────────
    if show_legend:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            bbox_to_anchor  = (legend_x, legend_y),
            ncol            = legend_cols,
            fontsize        = legend_fontsize,
            frameon         = False,
            loc             = "upper left",
        )

    plt.tight_layout()

    # ── 7. SAVE ───────────────────────────────────────────────────────────────
    if save_pdf or save_png:
        safe_which = which_var.replace(" ", "_")
        safe_var   = var.replace(" ", "_")
        stem = os.path.join(
            output_dir,
            f"{filename_prefix}{safe_which}_per_{safe_var}"
        )
        if save_pdf:
            fig.savefig(stem + ".pdf", bbox_inches="tight", dpi=dpi)
        if save_png:
            fig.savefig(stem + ".png", bbox_inches="tight", dpi=dpi)

    return fig, ax  # return both so caller can continue customising

In [ ]:
BARPLOT_COLS = [
    "celltype",
    # LEIDEN_COL
]

PROP_COL = "TISSUE_MERGED"

for col in BARPLOT_COLS:
    if col not in adata.obs.columns:
        print(f"Skipping '{col}' — not found in obs.")
        continue

    # Height scales with the number of bars (rows = unique values of col)
    _n_groups = adata.obs[col].nunique()
    _height   = max(2.0, _n_groups * 0.5)  # minimum 2 inches; 0.35 in per bar

    fig, ax = barplot(
        PROP_COL,
        adata,
        var             = col,
        height          = _height,
        width           = 10,
        bar_height      = 0.5,
        fontsize        = 12,
        legend_cols     = 1,
        legend_fontsize = 11,
        show_legend     = True,
    )

    fig.savefig(
        f"{OUTPUT_DIR}figures/barplot_{col}_{PROP_COL}_{OBJECT}.png",
        dpi=300, bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)  # free memory when looping over many columns

In [ ]:
BARPLOT_COLS = [
    "L3",
    # LEIDEN_COL
]

PROP_COL = "TISSUE_MERGED"

for col in BARPLOT_COLS:
    if col not in adata.obs.columns:
        print(f"Skipping '{col}' — not found in obs.")
        continue

    # Height scales with the number of bars (rows = unique values of col)
    _n_groups = adata.obs[col].nunique()
    _height   = max(2.0, _n_groups * 0.7)  # minimum 2 inches; 0.35 in per bar

    fig, ax = barplot(
        PROP_COL,
        adata,
        var             = col,
        height          = _height,
        width           = 10,
        bar_height      = 0.5,
        fontsize        = 12,
        legend_cols     = 1,
        legend_fontsize = 11,
        show_legend     = True,
    )

    fig.savefig(
        f"{OUTPUT_DIR}figures/barplot_{col}_{PROP_COL}_{OBJECT}.png",
        dpi=300, bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)  # free memory when looping over many columns

In [ ]:
# —— Broad tissue labels ———————————————————————————————————————————————————————
tissue_broad = {
    "Ovary": "ReproTract",
    "Fallopian Tube": "ReproTract",
    "Cervix": "ReproTract",
    "Vagina": "ReproTract",
    "Mesonephros": "ReproTract",
    "Reproductive tract": "ReproTract",
    
    "Endometrium": "ReproTract",
    "Myometrium": "ReproTract",
    "Whole Uterus": "ReproTract",
    "Menstrual fluid": "ReproTract",
    
    "Buccal": "NonReproTract",
    "Colon": "NonReproTract",
    "Gingiva": "NonReproTract",
    "Lung": "NonReproTract"
}

adata.obs["TISSUE_BROAD"] = adata.obs["TISSUE_MERGED"].map(tissue_broad)
# adata.obs["TISSUE_BROAD"].isna().sum()
adata.obs["TISSUE_BROAD"].value_counts()

In [ ]:
BARPLOT_COLS = [
    "celltype",
    # LEIDEN_COL
]

PROP_COL = "TISSUE_BROAD"

for col in BARPLOT_COLS:
    if col not in adata.obs.columns:
        print(f"Skipping '{col}' — not found in obs.")
        continue

    # Height scales with the number of bars (rows = unique values of col)
    _n_groups = adata.obs[col].nunique()
    _height   = max(2.0, _n_groups * 0.35)  # minimum 2 inches; 0.35 in per bar

    fig, ax = barplot(
        PROP_COL,
        adata,
        var             = col,
        height          = _height,
        width           = 10,
        bar_height      = 0.5,
        fontsize        = 12,
        legend_cols     = 1,
        legend_fontsize = 11,
        show_legend     = True,
    )

    fig.savefig(
        f"{OUTPUT_DIR}figures/barplot_{col}_{PROP_COL}_{OBJECT}.png",
        dpi=300, bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)  # free memory when looping over many columns

—— FIN ——